In [6]:
%pip install plotly

import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
import plotly.express as px

ROOT = Path.cwd().resolve()
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

a = pd.read_csv(OUT / "backtest_A_core_358.csv")
b = pd.read_csv(OUT / "backtest_B_price_only_500.csv")
c = pd.read_csv(OUT / "backtest_C_overlap.csv")

for df in [a, b, c]:
    df["report_date"] = pd.to_datetime(df["report_date"])
    df["cum_return"] = (1 + df["portfolio_return"].fillna(0)).cumprod() - 1

fig = go.Figure()
fig.add_trace(go.Scatter(x=a["report_date"], y=a["cum_return"], mode="lines", name="Model A"))
fig.add_trace(go.Scatter(x=b["report_date"], y=b["cum_return"], mode="lines", name="Model B"))
fig.add_trace(go.Scatter(x=c["report_date"], y=c["cum_return"], mode="lines", name="Model C"))
fig.update_layout(title="Cumulative Returns", xaxis_title="Date", yaxis_title="Cumulative Return")
fig.write_html(OUT / "cumulative_returns.html")

evals = pd.read_csv(OUT / "final_evaluation_summary.csv")
fig2 = px.bar(evals, x="model", y="sharpe_ratio", title="Sharpe Ratio by Model")
fig2.write_html(OUT / "sharpe_by_model.html")

fig3 = px.bar(evals, x="model", y="max_drawdown", title="Max Drawdown by Model")
fig3.write_html(OUT / "drawdown_by_model.html")

ols = pd.read_csv(OUT / "ols_all_summaries.csv")
fig4 = px.bar(ols[ols["term"] != "const"], x="term", y="Coef.", color="model", barmode="group", title="OLS Factor Coefficients by Model")
fig4.write_html(OUT / "ols_coefficients_by_model.html")

report = pd.DataFrame([
    {"artifact": "cumulative_returns.html", "purpose": "equity curves"},
    {"artifact": "sharpe_by_model.html", "purpose": "risk-adjusted comparison"},
    {"artifact": "drawdown_by_model.html", "purpose": "drawdown comparison"},
    {"artifact": "ols_coefficients_by_model.html", "purpose": "OLS factor coefficient comparison"},
])
report.to_csv(OUT / "report_artifacts.csv", index=False)

print(report)


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
                         artifact                            purpose
0         cumulative_returns.html                      equity curves
1            sharpe_by_model.html           risk-adjusted comparison
2          drawdown_by_model.html                drawdown comparison
3  ols_coefficients_by_model.html  OLS factor coefficient comparison
